In [ ]:
from PyLorentz.sim import SimLTEM
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PyLorentz.phase import TIE, SITIE
from PyLorentz.sim import SimLTEM
from PyLorentz.utils import Microscope
from PyLorentz.visualize import show_im, show_2D
import matplotlib.ticker as mticker

# load sample

In [ ]:
def read_vtk_magnetization(filename, vector_field_name=None, prefer="point"):
    """
    Read (x,y,z) positions and a 3-component magnetization vector from a VTK file.

    Parameters
    ----------
    filename : str
        Path to a .vtk/.vts/.vtu/.vti/... file readable by pyvista.
    vector_field_name : str, optional
        Name of the vector array to use. If None, auto-detected.
    prefer : {"point", "cell"}
        Whether to prefer point data or cell data when both are present.

    Returns
    -------
    points : (N, 3) ndarray
    M : (N, 3) ndarray
    mesh : the pyvista mesh object (kept around in case you need it, e.g.
           mesh.bounds, mesh.dimensions, other fields, etc.)
    """
    mesh = pv.read(filename)

    point_data = mesh.point_data
    cell_data = mesh.cell_data

    if prefer == "point" and len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    elif len(cell_data.keys()) > 0:
        data, points = cell_data, mesh.cell_centers().points
    elif len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    else:
        raise ValueError("No point_data or cell_data arrays found in file.")

    M = None
    if vector_field_name is not None:
        if vector_field_name not in data:
            raise ValueError(f"'{vector_field_name}' not found. Available: {list(data.keys())}")
        M = np.asarray(data[vector_field_name])
        if M.ndim != 2 or M.shape[1] != 3:
            raise ValueError(f"'{vector_field_name}' is not a 3-component vector array.")
    else:
        vector_candidates = [k for k in data.keys()
                              if np.asarray(data[k]).ndim == 2 and np.asarray(data[k]).shape[1] == 3]
        preferred_names = ["M", "Magnetization", "magnetization"]
        for name in preferred_names:
            if name in vector_candidates:
                M = np.asarray(data[name])
                break
        if M is None and len(vector_candidates) > 0:
            M = np.asarray(data[vector_candidates[0]])

        if M is None:
            component_sets = [("Mx", "My", "Mz"), ("mx", "my", "mz"),
                               ("M_x", "M_y", "M_z"), ("moment_x", "moment_y", "moment_z")]
            for cx, cy, cz in component_sets:
                if cx in data and cy in data and cz in data:
                    M = np.column_stack([np.asarray(data[cx]), np.asarray(data[cy]), np.asarray(data[cz])])
                    break

        if M is None:
            raise ValueError(
                "Could not auto-detect a magnetization vector field. "
                f"Available arrays: {list(data.keys())}. "
                "Pass vector_field_name explicitly."
            )

    return points, M, mesh

def _grid_from_points(points, M, decimals=8):
    """
    Reshape (N,3) points/M into full 3D grid arrays, using coordinate
    values (not assumed ordering) to place each point in its slot.
    """
    x = np.round(points[:, 0], decimals)
    y = np.round(points[:, 1], decimals)
    z = np.round(points[:, 2], decimals)

    x_u, y_u, z_u = np.unique(x), np.unique(y), np.unique(z)
    ix = np.searchsorted(x_u, x)
    iy = np.searchsorted(y_u, y)
    iz = np.searchsorted(z_u, z)

    grid_shape = (len(x_u), len(y_u), len(z_u))
    Mgrid = np.zeros(grid_shape + (3,))
    filled = np.zeros(grid_shape, dtype=bool)
    Mgrid[ix, iy, iz, :] = M
    filled[ix, iy, iz] = True

    if not filled.all():
        print(f"Warning: {(~filled).sum()} of {filled.size} grid slots "
              f"are empty and being treated as vacuum (M=0).")

    return x_u, y_u, z_u, Mgrid

def build_mags_array(points, M, normalize=True):
    """
    Convert an (N,3) point cloud + (N,3) magnetization into the
    (3, nz, ny, nx) array format SimLTEM expects.
    """
    x_u, y_u, z_u, Mgrid = _grid_from_points(points, M)  # shape (nx, ny, nz, 3)

    # reorder to (3, nz, ny, nx)
    mags = np.transpose(Mgrid, (3, 2, 1, 0))

    if normalize:
        mag_norm = np.linalg.norm(mags, axis=0, keepdims=True)
        mag_norm[mag_norm == 0] = 1.0  # avoid divide-by-zero in vacuum voxels
        mags = mags / mag_norm

    dx = x_u[1] - x_u[0]
    dy = y_u[1] - y_u[0]
    dz = z_u[1] - z_u[0] if len(z_u) > 1 else dx

    return mags, dx, dy, dz

In [ ]:
#filename = r'C:\Users\emmab\Projects\Vector_Tomography\452_mag_const_interpolated_cropped.vtk'\
filename = r'C:\Users\emmab\Projects\Vector_Tomography\270_mag_const_interpolated_cropped.vtk'

points, M, mesh = read_vtk_magnetization(filename)
print(f"Loaded {points.shape[0]} points, magnetization array shape {M.shape}")
print("Available point_data fields:", list(mesh.point_data.keys()))
print("Available cell_data fields:", list(mesh.cell_data.keys()))

# View loaded data and crop to ROI

In [ ]:
def average_through_thickness(points, M, nx=None, ny=None, thickness_axis=2):
    """
    Bin (x,y) and average M through the remaining axis (thickness_axis, default z).

    Parameters
    ----------
    points : (N,3) ndarray
    M : (N,3) ndarray
    nx, ny : int, optional
        Output grid resolution. If None, inferred from the number of
        unique coordinates along that axis (good for structured grids).
    thickness_axis : int
        Which column of `points` is the thickness direction to average
        over (0=x, 1=y, 2=z). Default 2 (z).

    Returns
    -------
    x_centers, y_centers : 1D ndarrays
    Mx_avg, My_avg, Mz_avg : 2D ndarrays, shape (ny, nx)
    """
    axes = [0, 1, 2]
    axes.remove(thickness_axis)
    ax_x, ax_y = axes

    x = points[:, ax_x]
    y = points[:, ax_y]

    x_unique = np.unique(np.round(x, decimals=8))
    y_unique = np.unique(np.round(y, decimals=8))

    if nx is None:
        nx = len(x_unique) if len(x_unique) > 1 else 100
    if ny is None:
        ny = len(y_unique) if len(y_unique) > 1 else 100

    x_edges = np.linspace(x.min(), x.max(), nx + 1)
    y_edges = np.linspace(y.min(), y.max(), ny + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    count, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    sums = []
    for comp in range(3):
        s, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges], weights=M[:, comp])
        sums.append(s)

    with np.errstate(invalid="ignore", divide="ignore"):
        avgs = [np.where(count > 0, s / count, np.nan) for s in sums]

    # transpose so array indexing is [row=y, col=x], matching imshow convention
    Mx_avg, My_avg, Mz_avg = [a.T for a in avgs]
    return x_centers, y_centers, Mx_avg, My_avg, Mz_avg


def crop_points(points, M, x_range=None, y_range=None, z_range=None):
    """
    Crop a point cloud + magnetization to a physical-coordinate bounding box.
    Any of x_range, y_range, z_range can be None to skip cropping that axis.
    Ranges are (min, max) tuples in the same units as `points`.
    """
    mask = np.ones(len(points), dtype=bool)
    if x_range is not None:
        mask &= (points[:, 0] >= x_range[0]) & (points[:, 0] <= x_range[1])
    if y_range is not None:
        mask &= (points[:, 1] >= y_range[0]) & (points[:, 1] <= y_range[1])
    if z_range is not None:
        mask &= (points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1])

    print(f"Cropped from {len(points)} to {mask.sum()} points "
          f"({100*mask.sum()/len(points):.1f}%).")
    return points[mask], M[mask]


def preview_crop_region(points, M, x_range=None, y_range=None, thickness_axis=2):
    """
    Plot the full through-thickness averaged Mx map with an optional
    rectangle overlay showing a proposed crop region, so you can check
    it captures the domain wall / interesting feature before cropping.
    """
    x_c, y_c, Mx_avg, My_avg, Mz_avg = average_through_thickness(
        points, M, thickness_axis=thickness_axis
    )

    fig, axs = plt.subplots(2, 1, figsize=(15, 6))
    for ax, data, label in zip(axs, [Mx_avg, My_avg], ["Mx (avg)", "My (avg)"]):
        im = ax.imshow(data, origin="lower",
                        extent=[x_c.min(), x_c.max(), y_c.min(), y_c.max()],
                        cmap="viridis")
        ax.set_title(label, font = 'Times New Roman')
        ax.set_xlabel('x (pix)', font = 'Times New Roman')
        ax.set_ylabel('y (pix)', font = 'Times New Roman')
        fig.colorbar(im, ax=ax, pad = 0.01 )

        if x_range is not None and y_range is not None:
            rect = patches.Rectangle(
                (x_range[0], y_range[0]),
                x_range[1] - x_range[0],
                y_range[1] - y_range[0],
                linewidth=2, edgecolor="k", facecolor="none"
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

def preview_crop_region_origin_none(points, M, x_range=None, y_range=None, thickness_axis=2):
    """
    Plot the full through-thickness averaged Mx map with an optional
    rectangle overlay showing a proposed crop region, so you can check
    it captures the domain wall / interesting feature before cropping.
    """
    x_c, y_c, Mx_avg, My_avg, Mz_avg = average_through_thickness(
        points, M, thickness_axis=thickness_axis
    )

    fig, axs = plt.subplots(2, 1, figsize=(15, 6))
    for ax, data, label in zip(axs, [Mx_avg, My_avg], ["Mx (avg)", "My (avg)"]):
        im = ax.imshow(data, #origin="lower",
                        extent=[x_c.min(), x_c.max(), y_c.min(), y_c.max()],
                        cmap="viridis")
        ax.set_title(label, font = 'Times New Roman')
        ax.set_xlabel('x (pix)', font = 'Times New Roman')
        ax.set_ylabel('y (pix)', font = 'Times New Roman')
        fig.colorbar(im, ax=ax, pad = 0.01 )

        if x_range is not None and y_range is not None:
            rect = patches.Rectangle(
                (x_range[0], y_range[0]),
                x_range[1] - x_range[0],
                y_range[1] - y_range[0],
                linewidth=2, edgecolor="k", facecolor="none"
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()


def plot_three_slice_fancy(mags, scale, zscale, field='Mz', label=None,
                            x_index=None, y_index=None, z_index=None,
                            cmap='viridis', vmin=None, vmax=None):
    """
    Plot an x-slice (y-z plane), y-slice (x-z plane), and z-slice (x-y plane)
    of `field` from a (3, nz, ny, nx) magnetization array, side by side.

    Parameters
    ----------
    mags : ndarray, shape (3, nz, ny, nx)
        Magnetization array, e.g. from build_mags_array / bin_mags.
    scale : float
        In-plane voxel size (nm/pixel), i.e. dx (assumes dx == dy).
    zscale : float
        Voxel size along z (nm/pixel).
    field : str
        Which field to view: 'Mx', 'My', 'Mz', or 'magnitude'
        (aliases 'mag'/'Mag' accepted, case-insensitive).
    x_index, y_index, z_index : int or None
        Voxel index for each slice. Defaults to the center of that axis.
    cmap : str
        Colormap for all three panels (default 'viridis').
    vmin, vmax : float or None
        Shared color limits across all three panels. If None, computed
        from the min/max of the full `field` array.

    Returns
    -------
    (fig, axes) : the created matplotlib figure and its 3 axes.
    """
    field_aliases = {'mag': 'magnitude', 'magnitude': 'magnitude',
                      'mx': 'Mx', 'my': 'My', 'mz': 'Mz'}
    key = field_aliases.get(field.lower())
    if key is None:
        raise KeyError(
            f"field '{field}' not recognized. "
            f"Expected one of: Mx, My, Mz, magnitude."
        )

    comp_idx = {'Mx': 0, 'My': 1, 'Mz': 2}
    if key == 'magnitude':
        data = np.linalg.norm(mags, axis=0)  # (nz, ny, nx)
    else:
        data = mags[comp_idx[key]]  # (nz, ny, nx)

    nz, ny, nx = data.shape

    if x_index is None:
        x_index = nx // 2
    if y_index is None:
        y_index = ny // 2
    if z_index is None:
        z_index = nz // 2

    if vmin is None:
        vmin = np.nanmin(data)
    if vmax is None:
        vmax = np.nanmax(data)

    slice_x = data[:, :, x_index]   # (nz, ny) -- y-z plane at fixed x
    slice_y = data[:, y_index, :]   # (nz, nx) -- x-z plane at fixed y
    slice_z = data[z_index, :, :]   # (ny, nx) -- x-y plane at fixed z

    LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 16}
    TITLE_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}
    TICK_FONT = {'family': 'Times New Roman', 'size': 13}
    CBAR_LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}

    def style_axis(ax):
        ax.set_xlabel(ax.get_xlabel(), fontdict=LABEL_FONT)
        ax.set_ylabel(ax.get_ylabel(), fontdict=LABEL_FONT)
        for lbl in ax.get_xticklabels() + ax.get_yticklabels():
            lbl.set_fontname('Times New Roman')
            lbl.set_fontsize(TICK_FONT['size'])

    def style_colorbar(cbar, label_text):
        cbar.set_label(label_text, fontdict=CBAR_LABEL_FONT)
        cbar.formatter = mticker.ScalarFormatter(useMathText=True)
        cbar.formatter.set_powerlimits((0, 0))
        cbar.update_ticks()
        for lbl in cbar.ax.get_yticklabels():
            lbl.set_fontname('Times New Roman')
            lbl.set_fontsize(TICK_FONT['size'])
        cbar.ax.yaxis.get_offset_text().set_fontname('Times New Roman')
        cbar.ax.yaxis.get_offset_text().set_fontsize(TICK_FONT['size'])

    # in-plane (x,y) extents shown in um, z extent shown in nm
    extent_x = [0, ny * scale / 1000, 0, nz * zscale]   # y (um), z (nm)
    extent_y = [0, nx * scale / 1000, 0, nz * zscale]   # x (um), z (nm)
    extent_z = [0, nx * scale / 1000, 0, ny * scale / 1000]  # x, y (um)

    fig = plt.figure(figsize=(25, 8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 2, 0.15], height_ratios=[1, 1],
                           hspace=0.37, wspace=0.15)

    # Z slice (left, full square)
    ax_z = fig.add_subplot(gs[:, 0])
    im_z = ax_z.imshow(slice_z, origin='lower', aspect='equal', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_z)
    ax_z.set_title("Z slice", fontdict=TITLE_FONT)
    ax_z.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_z.set_ylabel('y (μm)', fontdict=LABEL_FONT)
    ax_z.axhline(y_index * scale / 1000, color='magenta', lw=5, linestyle='--', zorder=3)
    ax_z.axvline(x_index * scale / 1000, color='darkorange', lw=5, linestyle='--', zorder=3)
    style_axis(ax_z)

    # X slice (top right, thin and long)
    ax_x = fig.add_subplot(gs[0, 1])
    im_x = ax_x.imshow(slice_x, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_x)
    ax_x.set_title("X slice", fontdict=TITLE_FONT, color='darkorange')
    ax_x.set_xlabel('y (μm)', fontdict=LABEL_FONT)
    ax_x.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_x)

    # Y slice (bottom right, thin and long)
    ax_y = fig.add_subplot(gs[1, 1])
    im_y = ax_y.imshow(slice_y, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_y)
    ax_y.set_title("Y slice", fontdict=TITLE_FONT, color='magenta')
    ax_y.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_y.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_y)

    cbar_xy = fig.colorbar(im_x, ax=[ax_x, ax_y], orientation='vertical',
                            pad=0.05, aspect=30)
    style_colorbar(cbar_xy, key)

    fig.suptitle(f"{key} Contrast at Volume Center \n {label}",
                 fontfamily=LABEL_FONT['family'],
                 fontweight=LABEL_FONT['weight'],
                 fontsize=20)

    plt.show()

    return fig, (ax_x, ax_y, ax_z)


In [ ]:
# start with a guess, adjust until the green box frames the wall well
plt.rcParams["font.family"] = "Times New Roman"

# x_range = (65, 95)
# y_range = (20, 60)

# # 452
# x_range = (25, 70)
# y_range = (55, 90)

# x_range = (60, 75)
# y_range = (80, 95)

# x_range = (30, 60)
# y_range = (30, 60)

# # 270
# x_range = (35, 65)
# y_range = (2, 32)

x_range = (0, 35)
y_range = (25, 75)

preview_crop_region(points, M, x_range=x_range, y_range=y_range)

mags_full, dx_full, dy_full, dz_full = build_mags_array(points, M)

fig, axes = plot_three_slice_fancy(
    mags_full, scale=dx_full, zscale=dz_full,
    field='Mz', label="Full volume"
)








## rotate 90

In [ ]:
# def rotate_mags_90(mags, k=1):
#     """
#     Rotate a (3, nz, ny, nx) [Mx, My, Mz] volume by k quarter-turns
#     counterclockwise (as displayed with origin='lower'), rotating both
#     positions and in-plane vectors. Exact: no interpolation.
#     """
#     k = k % 4
#     # axes=(3, 2) rotates from x toward y, i.e. counterclockwise with origin='lower'
#     out = np.rot90(mags, k=k, axes=(3, 2)).copy()
#     c, s = [(1, 0), (0, 1), (-1, 0), (0, -1)][k]   # cos, sin of k*90°
#     mx, my = out[0].copy(), out[1].copy()
#     out[0] = c * mx - s * my
#     out[1] = s * mx + c * my
#     print(f"Rotated by {90*k} deg: {mags.shape} -> {out.shape}")
#     return np.ascontiguousarray(out)

In [ ]:
# mags_r90 = rotate_mags_90(mags_full, k=1)
# fig, axs = plt.subplots(1, 2, figsize=(11, 5))
# plot_inplane(mags_full, dx, axs[0], "Original")
# plot_inplane(mags_r90, dx, axs[1], "Rotated +90°")
# plt.tight_layout(); plt.show()

## bin and crop

In [ ]:
def bin_mags(mags, bin_xy=2, bin_z=1, plot=False, dx=None, dy=None, dz=None):
    """
    Block-average (bin) a (3, nz, ny, nx) mags array to reduce resolution.

    Parameters
    ----------
    mags : ndarray, shape (3, nz, ny, nx)
    bin_xy : int
        Binning factor applied to both y and x (lateral) axes.
    bin_z : int
        Binning factor applied to z (thickness) axis. Usually 1 (no binning)
        since z is already coarse and linsup scales linearly in z anyway.
    plot : bool
        If True, show side-by-side Mx maps (through-thickness averaged)
        before and after binning for a sanity check.
    dx, dy, dz : float, optional
        Original voxel spacings; if given, returns updated spacings
        scaled by the bin factors.

    Returns
    -------
    mags_binned : ndarray
    (new_dx, new_dy, new_dz) : tuple of float, only if dx/dy/dz were given
    """
    _, nz, ny, nx = mags.shape

    nz_trim = nz - (nz % bin_z)
    ny_trim = ny - (ny % bin_xy)
    nx_trim = nx - (nx % bin_xy)
    if (nz_trim, ny_trim, nx_trim) != (nz, ny, nx):
        print(f"Trimming {nz,ny,nx} -> {nz_trim,ny_trim,nx_trim} "
              f"to be divisible by bin factors before binning.")
    mags_t = mags[:, :nz_trim, :ny_trim, :nx_trim]

    new_nz, new_ny, new_nx = nz_trim // bin_z, ny_trim // bin_xy, nx_trim // bin_xy
    reshaped = mags_t.reshape(3, new_nz, bin_z, new_ny, bin_xy, new_nx, bin_xy)
    mags_binned = reshaped.mean(axis=(2, 4, 6))

    mag_norm = np.linalg.norm(mags_binned, axis=0, keepdims=True)
    mag_norm_safe = np.where(mag_norm == 0, 1.0, mag_norm)
    mags_binned = mags_binned / mag_norm_safe

    print(f"Binned mags shape: {mags.shape} -> {mags_binned.shape} "
          f"({mags_binned.size / mags.size * 100:.1f}% of original voxels)")

    if plot:
        def through_thickness_Mx(m):
            return np.nanmean(m[0], axis=0)  # average over z, component 0 = Mx

        fig, axs = plt.subplots(1, 2, figsize=(9, 4))
        im0 = axs[0].imshow(through_thickness_Mx(mags), origin="lower", cmap="viridis")
        axs[0].set_title(f"Before binning {mags.shape[1:]}")
        fig.colorbar(im0, ax=axs[0], shrink=0.7)

        im1 = axs[1].imshow(through_thickness_Mx(mags_binned), origin="lower", cmap="viridis")
        axs[1].set_title(f"After binning {mags_binned.shape[1:]}")
        fig.colorbar(im1, ax=axs[1], shrink=0.7)
        plt.tight_layout()
        plt.show()

    if dx is not None and dy is not None and dz is not None:
        return mags_binned, (dx * bin_xy, dy * bin_xy, dz * bin_z)
    return mags_binned

In [ ]:
# Crop
points_crop, M_crop = crop_points(points, M, x_range=x_range, y_range=y_range)
preview_crop_region_origin_none(points_crop, M_crop, x_range=x_range, y_range=y_range)

In [ ]:
mags_crop, dx, dy, dz = build_mags_array(points_crop, M_crop)

mags_binned, (dx_b, dy_b, dz_b) = bin_mags(
    mags_crop, bin_xy=2, bin_z=1, plot=True, dx=dx, dy=dy, dz=dz
)
print("Binned shape (3, nz, ny, nx):", mags_binned.shape)

fig, axes = plot_three_slice_fancy(
    mags_binned, scale=dx_b, zscale=dz_b,
    field='Mx', label="binned volume"
)

ignore, nz, ny, nx = mags_binned.shape
print(nx * dx_b)

# Run a LTEM tilt series

In [ ]:
def to_pylorentz_order(mags):
    """
    Convert (3, nz, ny, nx) mags from [Mx, My, Mz] to PyLorentz's
    internal [Mz, My, Mx] component order. Returns a contiguous copy
    (a plain mags[::-1] view has negative strides, which numba may not like).
    """
    return np.ascontiguousarray(mags[::-1])

# ---------------------------------------------------------------------
# 2. Tilt series: compute phase + simulated LTEM images per angle
# ---------------------------------------------------------------------

def run_tilt_series(mags, scale, zscale, sample_params, beam_energy,
                     angles_deg, tilt_axis="x", defocus_values=(-100_000, 100_000),
                     scope_params=None, amorphous_bkg=True, verbose=0):
    """
    Run compute_phase + sim_images across a range of tilt angles.

    Parameters
    ----------
    mags, scale, zscale : inputs for SimLTEM
    sample_params : dict, passed to set_sample_params
    beam_energy : float, eV
    angles_deg : iterable of float, e.g. np.arange(-30, 31, 10)
    tilt_axis : "x" or "y"
    defocus_values : list/tuple of defocus values, nm
    scope_params : dict, optional overrides for Microscope (Cs, theta_c, def_spr, etc.)
    amorphous_bkg : passed to sim_images

    Returns
    -------
    results : dict keyed by angle, each value a dict with keys
        "phase_B", "shape_func", "images", "defocus_values", "scale"
    """
    scope_defaults = dict(E=beam_energy, Cs=200e3, theta_c=0.01e-3, def_spr=80)
    if scope_params:
        scope_defaults.update(scope_params)
    scope = Microscope(**scope_defaults, verbose=False)

    results = {}
    for theta in angles_deg:
        sim = SimLTEM(mags=to_pylorentz_order(mags), scale=scale, zscale=zscale, verbose=verbose)
        sim.set_sample_params(sample_params)

        kwargs = {"tilt_x": float(theta), "tilt_y": 0.0} if tilt_axis == "x" \
            else {"tilt_x": 0.0, "tilt_y": float(theta)}

        sim.compute_phase(method="linsup", beam_energy=beam_energy, **kwargs)

        dd = sim.sim_images(defocus_values=list(defocus_values), scope=scope,
                             amorphous_bkg=amorphous_bkg)

        results[theta] = {
            "phase_B": sim.phase_B,
            "shape_func": sim.shape_func,
            "images": dd.images,
            "defocus_values": list(defocus_values),
            "scale": sim.scale,
        }
        print(f"tilt = {theta:+.0f}deg done.")

    return results


# ---------------------------------------------------------------------
# 3. Plotting
# ---------------------------------------------------------------------

def plot_phase_tilt_series(results, cmap="RdBu_r"):
    """Grid plot of phase_B across all tilt angles, shared color scale."""
    angles = sorted(results.keys())
    vmax = max(np.abs(results[a]["phase_B"]).max() for a in angles)

    ncols = min(len(angles), 5)
    nrows = int(np.ceil(len(angles) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    for i, theta in enumerate(angles):
        ax = axs[i // ncols][i % ncols]
        im = ax.imshow(results[theta]["phase_B"], origin="lower",
                        cmap=cmap, vmin=-vmax, vmax=vmax)
        ax.set_title(f"{theta:+.0f}°")
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(len(angles), nrows*ncols):
        axs[j // ncols][j % ncols].axis("off")

    fig.colorbar(im, ax=axs, shrink=0.6, label="Phase (rad)")
    fig.suptitle("Phase shift (phase_B) vs. tilt angle")
    plt.show()


# def plot_LTEM_tilt_series(results, defocus_index=0, cmap="gray"):
#     """
#     Grid plot of simulated LTEM images across all tilt angles, for one
#     chosen defocus value (by index into the defocus_values list used).
#     """
#     angles = sorted(results.keys())
#     defval = results[angles[0]]["defocus_values"][defocus_index]

#     all_ims = [results[a]["images"][defocus_index] for a in angles]
#     vmin = min(im.min() for im in all_ims)
#     vmax = max(im.max() for im in all_ims)

#     nrows = min(len(angles), 5)
#     ncols = int(np.ceil(len(angles) / nrows))
#     fig, axs = plt.subplots(nrows, ncols, figsize=(3*ncols, 2*nrows), squeeze=False) #squeeze=False

#     for i, theta in enumerate(angles):
#         ax = axs[i // ncols][i % ncols]
#         im = ax.imshow(results[theta]["images"][defocus_index], origin="lower",
#                         cmap=cmap, vmin=vmin, vmax=vmax)
#         ax.set_title(f"{theta:+.0f}°")
#         ax.set_xticks([]); ax.set_yticks([])

#     for j in range(len(angles), nrows*ncols):
#         axs[j // ncols][j % ncols].axis("off")

#     fig.colorbar(im, ax=axs,  label="Intensity", shrink=0.6, pad = 0.2) #
#     fig.suptitle(f"Simulated LTEM image (PyLorentz) vs. tilt angle ")#, y = 0.95) \n (defocus = {defval:.0f} nm)
#     plt.show()


def plot_LTEM_tilt_series_horiz(results, defocus_index=0, cmap="gray"):
    """
    Grid plot of simulated LTEM images across all tilt angles, for one
    chosen defocus value (by index into the defocus_values list used).
    """
    angles = sorted(results.keys())
    defval = results[angles[0]]["defocus_values"][defocus_index]

    all_ims = [results[a]["images"][defocus_index] for a in angles]
    vmin = min(im.min() for im in all_ims)
    vmax = max(im.max() for im in all_ims)

    ncols = min(len(angles), 5)
    nrows = int(np.ceil(len(angles) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3*ncols, 2*nrows), squeeze=False) #squeeze=False

    for i, theta in enumerate(angles):
        ax = axs[i // ncols][i % ncols]
        im = ax.imshow(results[theta]["images"][defocus_index], origin="lower",
                        cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"{theta:+.0f}°")
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(len(angles), nrows*ncols):
        axs[j // ncols][j % ncols].axis("off")

    fig.colorbar(im, ax=axs) #, shrink=0.7) #, pad = 0.1) #
    #fig.suptitle(f"Simulated LTEM image (PyLorentz) vs. tilt angle ")#, y = 0.95) \n (defocus = {defval:.0f} nm)
    plt.show()

## set up sample and microscope parameters

In [ ]:
sample_params = {
    "B0": 8708,             # Gauss from VSM
    "sample_xip0": 100,     # nm, placeholder
    "sample_V0": 0,
    "membrane_thickness": 0,
}
beam_energy = 300e3  # eV

# 3. run the tilt series
angles_deg = np.arange(-30, 31, 30)


## test wall

In [ ]:
def make_test_wall(nz=8, ny=128, nx=128, width_px=4, psi_deg=0):
    """
    PMA film with one straight wall running along x (wall normal along y),
    i.e. parallel to the tilt_x axis.
    psi_deg = 0  -> pure Néel (in-plane M along wall normal, y)
    psi_deg = 90 -> pure Bloch (in-plane M along the wall, x)
    in between   -> hybrid
    """
    y = np.arange(ny) - ny / 2
    theta = 2 * np.arctan(np.exp(y / width_px))   # Mz goes +1 -> -1 across the wall
    mz, m_ip = np.cos(theta), np.sin(theta)
    psi = np.deg2rad(psi_deg)
    mags = np.zeros((3, nz, ny, nx))
    mags[0] = (m_ip * np.sin(psi))[None, :, None]   # Mx
    mags[1] = (m_ip * np.cos(psi))[None, :, None]   # My
    mags[2] = mz[None, :, None]                     # Mz
    return mags

for psi, name in [(0, "Néel"), (90, "Bloch"), (30, "Hybrid, Néel-leaning")]:
    test_mags = make_test_wall(psi_deg=psi)
    res = run_tilt_series(test_mags, scale=2, zscale=2, sample_params=sample_params,
                          beam_energy=beam_energy, angles_deg=[-20, 0, 20],
                          tilt_axis="y", defocus_values=[-100_000])
    print(name)
    plot_LTEM_tilt_series(res, defocus_index=0)

In [ ]:
def uniform_slab(component, nz=8, ny=64, nx=64):
    mags = np.zeros((3, nz, ny, nx))
    mags[component] = 1.0
    return mags

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for i, label in enumerate(["mags[0] = 1", "mags[1] = 1", "mags[2] = 1"]):
    sim = SimLTEM(mags=uniform_slab(i), scale=2, zscale=2, verbose=0)
    sim.set_sample_params(sample_params)
    sim.compute_phase(method="linsup", tilt_x=0, tilt_y=0, beam_energy=beam_energy)
    im = axs[i].imshow(sim.phase_B, origin="lower", cmap="RdBu_r")
    axs[i].set_title(label)
    fig.colorbar(im, ax=axs[i], shrink=0.7)
plt.tight_layout()
plt.show()

## defocus test

In [ ]:
def suggest_defocus(phase, scale, beam_energy, desired_fringe_width_nm=None, verbose=True):
    """
    Estimate a reasonable defocus based on the actual computed phase gradient,
    so Fresnel contrast is visible but doesn't exceed the field of view.

    phase : 2D ndarray, e.g. simulator.phase_B (radians)
    scale : float, in-plane nm/pixel
    beam_energy : float, eV
    desired_fringe_width_nm : float, optional
        Target width of the Fresnel fringe in nm. Defaults to ~10 pixels.
    """
    V = beam_energy  # volts
    lam_nm = 1.22639 / np.sqrt(V + 0.97845e-6 * V**2)  # relativistic electron wavelength, nm

    dphidy, dphidx = np.gradient(phase, scale)  # rad/nm
    grad_mag = np.sqrt(dphidx**2 + dphidy**2)
    max_grad = np.nanmax(grad_mag)

    beta = lam_nm / (2*np.pi) * max_grad  # beam deflection angle, radians

    if desired_fringe_width_nm is None:
        desired_fringe_width_nm = 10 * scale  # default target: ~10 pixels wide

    defocus_nm = desired_fringe_width_nm / beta if beta > 0 else np.inf

    if verbose:
        print(f"Electron wavelength: {lam_nm:.5f} nm")
        print(f"Max phase gradient: {max_grad:.4e} rad/nm")
        print(f"Max beam deflection angle: {beta:.4e} rad")
        print(f"Suggested defocus for ~{desired_fringe_width_nm:.1f} nm fringe: "
              f"{defocus_nm:.0f} nm ({defocus_nm/1000:.2f} um)")

    return defocus_nm

In [ ]:
defocus_test = SimLTEM(mags=mags_crop, scale=dx, zscale=dz)
defocus_test.set_sample_params(sample_params)

tilt_x=0

import time
t0 = time.time()
defocus_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=0, beam_energy=beam_energy)
t1 = time.time()
print(f"Time for cropped volume: {t1 - t0:.2f} sec")

defocus_est = suggest_defocus(defocus_test.phase_B, scale=defocus_test.scale, beam_energy=beam_energy)
defocus = defocus_est/ 1000
#defocus = 1620.99  #1086.72 

## quick glance

In [ ]:
results_xtilt = run_tilt_series(
    mags_binned, dx_b, dz_b, sample_params, beam_energy,
    angles_deg=angles_deg, tilt_axis="x",
    defocus_values=[-defocus, defocus],
)

# 4. plot
# plot_phase_tilt_series(results)
plot_LTEM_tilt_series(results_xtilt, defocus_index=0)  # underfocus


results_ytilt = run_tilt_series(
    mags_binned, dx_b, dz_b, sample_params, beam_energy,
    angles_deg=angles_deg, tilt_axis="y",
    defocus_values=[-defocus, defocus],
)
plot_LTEM_tilt_series(results_ytilt, defocus_index=0)  # overfocus

In [ ]:
plot_LTEM_tilt_series_horiz(results_xtilt, defocus_index=0)  # underfocus

plot_LTEM_tilt_series_horiz(results_ytilt, defocus_index=0)  # overfocus

In [ ]:
stop

# look at individual runs 

In [ ]:
def average_through_thickness(points, M, nx=None, ny=None, thickness_axis=2):
    """
    Bin (x,y) and average M through the remaining axis (thickness_axis, default z).

    Parameters
    ----------
    points : (N,3) ndarray
    M : (N,3) ndarray
    nx, ny : int, optional
        Output grid resolution. If None, inferred from the number of
        unique coordinates along that axis (good for structured grids).
    thickness_axis : int
        Which column of `points` is the thickness direction to average
        over (0=x, 1=y, 2=z). Default 2 (z).

    Returns
    -------
    x_centers, y_centers : 1D ndarrays
    Mx_avg, My_avg, Mz_avg : 2D ndarrays, shape (ny, nx)
    """
    axes = [0, 1, 2]
    axes.remove(thickness_axis)
    ax_x, ax_y = axes

    x = points[:, ax_x]
    y = points[:, ax_y]

    x_unique = np.unique(np.round(x, decimals=8))
    y_unique = np.unique(np.round(y, decimals=8))

    if nx is None:
        nx = len(x_unique) if len(x_unique) > 1 else 100
    if ny is None:
        ny = len(y_unique) if len(y_unique) > 1 else 100

    x_edges = np.linspace(x.min(), x.max(), nx + 1)
    y_edges = np.linspace(y.min(), y.max(), ny + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    count, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    sums = []
    for comp in range(3):
        s, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges], weights=M[:, comp])
        sums.append(s)

    with np.errstate(invalid="ignore", divide="ignore"):
        avgs = [np.where(count > 0, s / count, np.nan) for s in sums]

    # transpose so array indexing is [row=y, col=x], matching imshow convention
    Mx_avg, My_avg, Mz_avg = [a.T for a in avgs]
    return x_centers, y_centers, Mx_avg, My_avg, Mz_avg

In [ ]:
def read_vtk_magnetization(filename, vector_field_name=None, prefer="point"):
    """
    Read (x,y,z) positions and a 3-component magnetization vector from a VTK file.

    Parameters
    ----------
    filename : str
        Path to a .vtk/.vts/.vtu/.vti/... file readable by pyvista.
    vector_field_name : str, optional
        Name of the vector array to use. If None, auto-detected.
    prefer : {"point", "cell"}
        Whether to prefer point data or cell data when both are present.

    Returns
    -------
    points : (N, 3) ndarray
    M : (N, 3) ndarray
    mesh : the pyvista mesh object (kept around in case you need it, e.g.
           mesh.bounds, mesh.dimensions, other fields, etc.)
    """
    mesh = pv.read(filename)

    point_data = mesh.point_data
    cell_data = mesh.cell_data

    if prefer == "point" and len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    elif len(cell_data.keys()) > 0:
        data, points = cell_data, mesh.cell_centers().points
    elif len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    else:
        raise ValueError("No point_data or cell_data arrays found in file.")

    M = None
    if vector_field_name is not None:
        if vector_field_name not in data:
            raise ValueError(f"'{vector_field_name}' not found. Available: {list(data.keys())}")
        M = np.asarray(data[vector_field_name])
        if M.ndim != 2 or M.shape[1] != 3:
            raise ValueError(f"'{vector_field_name}' is not a 3-component vector array.")
    else:
        vector_candidates = [k for k in data.keys()
                              if np.asarray(data[k]).ndim == 2 and np.asarray(data[k]).shape[1] == 3]
        preferred_names = ["M", "Magnetization", "magnetization", "moment", "spin", "Spin"]
        for name in preferred_names:
            if name in vector_candidates:
                M = np.asarray(data[name])
                break
        if M is None and len(vector_candidates) > 0:
            M = np.asarray(data[vector_candidates[0]])

        if M is None:
            component_sets = [("Mx", "My", "Mz"), ("mx", "my", "mz"),
                               ("M_x", "M_y", "M_z"), ("moment_x", "moment_y", "moment_z")]
            for cx, cy, cz in component_sets:
                if cx in data and cy in data and cz in data:
                    M = np.column_stack([np.asarray(data[cx]), np.asarray(data[cy]), np.asarray(data[cz])])
                    break

        if M is None:
            raise ValueError(
                "Could not auto-detect a magnetization vector field. "
                f"Available arrays: {list(data.keys())}. "
                "Pass vector_field_name explicitly."
            )

    return points, M, mesh

def _grid_from_points(points, M, decimals=8):
    """
    Reshape (N,3) points/M into full 3D grid arrays, using coordinate
    values (not assumed ordering) to place each point in its slot.
    """
    x = np.round(points[:, 0], decimals)
    y = np.round(points[:, 1], decimals)
    z = np.round(points[:, 2], decimals)

    x_u, y_u, z_u = np.unique(x), np.unique(y), np.unique(z)
    ix = np.searchsorted(x_u, x)
    iy = np.searchsorted(y_u, y)
    iz = np.searchsorted(z_u, z)

    grid_shape = (len(x_u), len(y_u), len(z_u))
    Mgrid = np.zeros(grid_shape + (3,))
    filled = np.zeros(grid_shape, dtype=bool)
    Mgrid[ix, iy, iz, :] = M
    filled[ix, iy, iz] = True

    if not filled.all():
        print(f"Warning: {(~filled).sum()} of {filled.size} grid slots "
              f"are empty and being treated as vacuum (M=0).")

    return x_u, y_u, z_u, Mgrid

def build_mags_array(points, M, normalize=True):
    """
    Convert an (N,3) point cloud + (N,3) magnetization into the
    (3, nz, ny, nx) array format SimLTEM expects.
    """
    x_u, y_u, z_u, Mgrid = _grid_from_points(points, M)  # shape (nx, ny, nz, 3)

    # reorder to (3, nz, ny, nx)
    mags = np.transpose(Mgrid, (3, 2, 1, 0))

    if normalize:
        mag_norm = np.linalg.norm(mags, axis=0, keepdims=True)
        mag_norm[mag_norm == 0] = 1.0  # avoid divide-by-zero in vacuum voxels
        mags = mags / mag_norm

    dx = x_u[1] - x_u[0]
    dy = y_u[1] - y_u[0]
    dz = z_u[1] - z_u[0] if len(z_u) > 1 else dx

    return mags, dx, dy, dz

In [ ]:
filename = r'C:\Users\emmab\Projects\Vector_Tomography\452_mag_const_interpolated_cropped.vtk'

points, M, mesh = read_vtk_magnetization(filename)
print(f"Loaded {points.shape[0]} points, magnetization array shape {M.shape}")
print("Available point_data fields:", list(mesh.point_data.keys()))
print("Available cell_data fields:", list(mesh.cell_data.keys()))

In [ ]:
def crop_points(points, M, x_range=None, y_range=None, z_range=None):
    """
    Crop a point cloud + magnetization to a physical-coordinate bounding box.
    Any of x_range, y_range, z_range can be None to skip cropping that axis.
    Ranges are (min, max) tuples in the same units as `points`.
    """
    mask = np.ones(len(points), dtype=bool)
    if x_range is not None:
        mask &= (points[:, 0] >= x_range[0]) & (points[:, 0] <= x_range[1])
    if y_range is not None:
        mask &= (points[:, 1] >= y_range[0]) & (points[:, 1] <= y_range[1])
    if z_range is not None:
        mask &= (points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1])

    print(f"Cropped from {len(points)} to {mask.sum()} points "
          f"({100*mask.sum()/len(points):.1f}%).")
    return points[mask], M[mask]


import matplotlib.pyplot as plt
import matplotlib.patches as patches

def preview_crop_region(points, M, x_range=None, y_range=None, thickness_axis=2):
    """
    Plot the full through-thickness averaged Mx map with an optional
    rectangle overlay showing a proposed crop region, so you can check
    it captures the domain wall / interesting feature before cropping.
    """
    x_c, y_c, Mx_avg, My_avg, Mz_avg = average_through_thickness(
        points, M, thickness_axis=thickness_axis
    )

    fig, axs = plt.subplots(2, 1, figsize=(15, 6))
    for ax, data, label in zip(axs, [Mx_avg, My_avg], ["Mx (avg)", "My (avg)"]):
        im = ax.imshow(data, origin="lower",
                        extent=[x_c.min(), x_c.max(), y_c.min(), y_c.max()],
                        cmap="viridis")
        ax.set_title(label, font = 'Times New Roman')
        ax.set_xlabel('x (pix)', font = 'Times New Roman')
        ax.set_ylabel('y (pix)', font = 'Times New Roman')
        fig.colorbar(im, ax=ax, pad = 0.01 )

        if x_range is not None and y_range is not None:
            rect = patches.Rectangle(
                (x_range[0], y_range[0]),
                x_range[1] - x_range[0],
                y_range[1] - y_range[0],
                linewidth=2, edgecolor="k", facecolor="none"
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()


def preview_crop_region_origin_none(points, M, x_range=None, y_range=None, thickness_axis=2):
    """
    Plot the full through-thickness averaged Mx map with an optional
    rectangle overlay showing a proposed crop region, so you can check
    it captures the domain wall / interesting feature before cropping.
    """
    x_c, y_c, Mx_avg, My_avg, Mz_avg = average_through_thickness(
        points, M, thickness_axis=thickness_axis
    )

    fig, axs = plt.subplots(2, 1, figsize=(15, 6))
    for ax, data, label in zip(axs, [Mx_avg, My_avg], ["Mx (avg)", "My (avg)"]):
        im = ax.imshow(data, #origin="lower",
                        extent=[x_c.min(), x_c.max(), y_c.min(), y_c.max()],
                        cmap="viridis")
        ax.set_title(label, font = 'Times New Roman')
        ax.set_xlabel('x (pix)', font = 'Times New Roman')
        ax.set_ylabel('y (pix)', font = 'Times New Roman')
        fig.colorbar(im, ax=ax, pad = 0.01 )

        if x_range is not None and y_range is not None:
            rect = patches.Rectangle(
                (x_range[0], y_range[0]),
                x_range[1] - x_range[0],
                y_range[1] - y_range[0],
                linewidth=2, edgecolor="k", facecolor="none"
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()

In [ ]:
# start with a guess, adjust until the green box frames the wall well
plt.rcParams["font.family"] = "Times New Roman"

# x_range = (65, 95)
# y_range = (20, 60)

x_range = (40, 60)
y_range = (60, 80)
preview_crop_region(points, M, x_range=x_range, y_range=y_range)

In [ ]:
points_crop, M_crop = crop_points(points, M, x_range=x_range, y_range=y_range)

In [ ]:
def bin_mags(mags, bin_xy=2, bin_z=1, plot=False, dx=None, dy=None, dz=None):
    """
    Block-average (bin) a (3, nz, ny, nx) mags array to reduce resolution.

    Parameters
    ----------
    mags : ndarray, shape (3, nz, ny, nx)
    bin_xy : int
        Binning factor applied to both y and x (lateral) axes.
    bin_z : int
        Binning factor applied to z (thickness) axis. Usually 1 (no binning)
        since z is already coarse and linsup scales linearly in z anyway.
    plot : bool
        If True, show side-by-side Mx maps (through-thickness averaged)
        before and after binning for a sanity check.
    dx, dy, dz : float, optional
        Original voxel spacings; if given, returns updated spacings
        scaled by the bin factors.

    Returns
    -------
    mags_binned : ndarray
    (new_dx, new_dy, new_dz) : tuple of float, only if dx/dy/dz were given
    """
    _, nz, ny, nx = mags.shape

    nz_trim = nz - (nz % bin_z)
    ny_trim = ny - (ny % bin_xy)
    nx_trim = nx - (nx % bin_xy)
    if (nz_trim, ny_trim, nx_trim) != (nz, ny, nx):
        print(f"Trimming {nz,ny,nx} -> {nz_trim,ny_trim,nx_trim} "
              f"to be divisible by bin factors before binning.")
    mags_t = mags[:, :nz_trim, :ny_trim, :nx_trim]

    new_nz, new_ny, new_nx = nz_trim // bin_z, ny_trim // bin_xy, nx_trim // bin_xy
    reshaped = mags_t.reshape(3, new_nz, bin_z, new_ny, bin_xy, new_nx, bin_xy)
    mags_binned = reshaped.mean(axis=(2, 4, 6))

    mag_norm = np.linalg.norm(mags_binned, axis=0, keepdims=True)
    mag_norm_safe = np.where(mag_norm == 0, 1.0, mag_norm)
    mags_binned = mags_binned / mag_norm_safe

    print(f"Binned mags shape: {mags.shape} -> {mags_binned.shape} "
          f"({mags_binned.size / mags.size * 100:.1f}% of original voxels)")

    if plot:
        def through_thickness_Mx(m):
            return np.nanmean(m[0], axis=0)  # average over z, component 0 = Mx

        fig, axs = plt.subplots(1, 2, figsize=(9, 4))
        im0 = axs[0].imshow(through_thickness_Mx(mags), origin="lower", cmap="viridis")
        axs[0].set_title(f"Before binning {mags.shape[1:]}")
        fig.colorbar(im0, ax=axs[0], shrink=0.7)

        im1 = axs[1].imshow(through_thickness_Mx(mags_binned), origin="lower", cmap="viridis")
        axs[1].set_title(f"After binning {mags_binned.shape[1:]}")
        fig.colorbar(im1, ax=axs[1], shrink=0.7)
        plt.tight_layout()
        plt.show()

    if dx is not None and dy is not None and dz is not None:
        return mags_binned, (dx * bin_xy, dy * bin_xy, dz * bin_z)
    return mags_binned

In [ ]:
mags_crop, dx, dy, dz = build_mags_array(points_crop, M_crop)

mags_binned, (dx_b, dy_b, dz_b) = bin_mags(
    mags_crop, bin_xy=2, bin_z=1, plot=True, dx=dx, dy=dy, dz=dz
)
print("Binned shape (3, nz, ny, nx):", mags_binned.shape)

In [ ]:
preview_crop_region_origin_none(points_crop, M_crop, x_range=x_range, y_range=y_range)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PyLorentz.sim import SimLTEM
from PyLorentz.utils import Microscope


# ---------------------------------------------------------------------
# 1. Data prep: crop, grid, bin
# ---------------------------------------------------------------------

def crop_points(points, M, x_range=None, y_range=None, z_range=None):
    """Crop a point cloud + magnetization to a physical bounding box."""
    mask = np.ones(len(points), dtype=bool)
    if x_range is not None:
        mask &= (points[:, 0] >= x_range[0]) & (points[:, 0] <= x_range[1])
    if y_range is not None:
        mask &= (points[:, 1] >= y_range[0]) & (points[:, 1] <= y_range[1])
    if z_range is not None:
        mask &= (points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1])
    print(f"Cropped from {len(points)} to {mask.sum()} points "
          f"({100*mask.sum()/len(points):.1f}%).")
    return points[mask], M[mask]


def _grid_from_points(points, M, decimals=8):
    """Reshape (N,3) points/M into full 3D grid arrays (nx, ny, nz, 3)."""
    x = np.round(points[:, 0], decimals)
    y = np.round(points[:, 1], decimals)
    z = np.round(points[:, 2], decimals)
    x_u, y_u, z_u = np.unique(x), np.unique(y), np.unique(z)
    ix = np.searchsorted(x_u, x)
    iy = np.searchsorted(y_u, y)
    iz = np.searchsorted(z_u, z)

    grid_shape = (len(x_u), len(y_u), len(z_u))
    Mgrid = np.zeros(grid_shape + (3,))
    filled = np.zeros(grid_shape, dtype=bool)
    Mgrid[ix, iy, iz, :] = M
    filled[ix, iy, iz] = True

    if not filled.all():
        print(f"Warning: {(~filled).sum()} of {filled.size} grid slots "
              f"are empty and treated as vacuum (M=0).")
    return x_u, y_u, z_u, Mgrid


def build_mags_array(points, M, normalize=True):
    """Convert (N,3) points + M into (3, nz, ny, nx) array for SimLTEM."""
    x_u, y_u, z_u, Mgrid = _grid_from_points(points, M)
    mags = np.transpose(Mgrid, (3, 2, 1, 0))  # (3, nz, ny, nx)

    if normalize:
        mag_norm = np.linalg.norm(mags, axis=0, keepdims=True)
        mag_norm[mag_norm == 0] = 1.0
        mags = mags / mag_norm

    dx = x_u[1] - x_u[0]
    dy = y_u[1] - y_u[0]
    dz = z_u[1] - z_u[0] if len(z_u) > 1 else dx
    return mags, dx, dy, dz


def bin_mags(mags, bin_xy=2, bin_z=1, dx=None, dy=None, dz=None,
             plot=False, plot_components=("Mx", "My", "Mz")):
    """Block-average (bin) a (3, nz, ny, nx) mags array."""
    _, nz, ny, nx = mags.shape
    nz_trim = nz - (nz % bin_z)
    ny_trim = ny - (ny % bin_xy)
    nx_trim = nx - (nx % bin_xy)
    mags_t = mags[:, :nz_trim, :ny_trim, :nx_trim]

    new_nz, new_ny, new_nx = nz_trim // bin_z, ny_trim // bin_xy, nx_trim // bin_xy
    reshaped = mags_t.reshape(3, new_nz, bin_z, new_ny, bin_xy, new_nx, bin_xy)
    mags_binned = reshaped.mean(axis=(2, 4, 6))

    mag_norm = np.linalg.norm(mags_binned, axis=0, keepdims=True)
    mag_norm_safe = np.where(mag_norm == 0, 1.0, mag_norm)
    mags_binned = mags_binned / mag_norm_safe

    print(f"Binned mags shape: {mags.shape} -> {mags_binned.shape} "
          f"({mags_binned.size / mags.size * 100:.1f}% of original voxels)")

    if plot:
        comp_idx = {"Mx": 0, "My": 1, "Mz": 2}
        n = len(plot_components)
        fig, axs = plt.subplots(2, n, figsize=(3*n, 6))
        for i, comp in enumerate(plot_components):
            idx = comp_idx[comp]
            before = np.nanmean(mags[idx], axis=0)
            after = np.nanmean(mags_binned[idx], axis=0)
            vmax = max(np.abs(before).max(), np.abs(after).max())
            axs[0, i].imshow(before, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            axs[0, i].set_title(f"{comp} before")
            axs[1, i].imshow(after, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            axs[1, i].set_title(f"{comp} after")
        plt.tight_layout()
        plt.show()

    if dx is not None and dy is not None and dz is not None:
        return mags_binned, (dx * bin_xy, dy * bin_xy, dz * bin_z)
    return mags_binned


# ---------------------------------------------------------------------
# 2. Tilt series: compute phase + simulated LTEM images per angle
# ---------------------------------------------------------------------

def run_tilt_series(mags, scale, zscale, sample_params, beam_energy,
                     angles_deg, tilt_axis="x", defocus_values=(-100_000, 100_000),
                     scope_params=None, amorphous_bkg=True, verbose=0):
    """
    Run compute_phase + sim_images across a range of tilt angles.

    Parameters
    ----------
    mags, scale, zscale : inputs for SimLTEM
    sample_params : dict, passed to set_sample_params
    beam_energy : float, eV
    angles_deg : iterable of float, e.g. np.arange(-30, 31, 10)
    tilt_axis : "x" or "y"
    defocus_values : list/tuple of defocus values, nm
    scope_params : dict, optional overrides for Microscope (Cs, theta_c, def_spr, etc.)
    amorphous_bkg : passed to sim_images

    Returns
    -------
    results : dict keyed by angle, each value a dict with keys
        "phase_B", "shape_func", "images", "defocus_values", "scale"
    """
    scope_defaults = dict(E=beam_energy, Cs=200e3, theta_c=0.01e-3, def_spr=80)
    if scope_params:
        scope_defaults.update(scope_params)
    scope = Microscope(**scope_defaults, verbose=False)

    results = {}
    for theta in angles_deg:
        sim = SimLTEM(mags=mags, scale=scale, zscale=zscale, verbose=verbose)
        sim.set_sample_params(sample_params)

        kwargs = {"tilt_x": float(theta), "tilt_y": 0.0} if tilt_axis == "x" \
            else {"tilt_x": 0.0, "tilt_y": float(theta)}

        sim.compute_phase(method="linsup", beam_energy=beam_energy, **kwargs)

        dd = sim.sim_images(defocus_values=list(defocus_values), scope=scope,
                             amorphous_bkg=amorphous_bkg)

        results[theta] = {
            "phase_B": sim.phase_B,
            "shape_func": sim.shape_func,
            "images": dd.images,
            "defocus_values": list(defocus_values),
            "scale": sim.scale,
        }
        print(f"tilt = {theta:+.0f}deg done.")

    return results


# ---------------------------------------------------------------------
# 3. Plotting
# ---------------------------------------------------------------------

def plot_phase_tilt_series(results, cmap="RdBu_r"):
    """Grid plot of phase_B across all tilt angles, shared color scale."""
    angles = sorted(results.keys())
    vmax = max(np.abs(results[a]["phase_B"]).max() for a in angles)

    ncols = min(len(angles), 5)
    nrows = int(np.ceil(len(angles) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    for i, theta in enumerate(angles):
        ax = axs[i // ncols][i % ncols]
        im = ax.imshow(results[theta]["phase_B"], origin="lower",
                        cmap=cmap, vmin=-vmax, vmax=vmax)
        ax.set_title(f"{theta:+.0f}°")
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(len(angles), nrows*ncols):
        axs[j // ncols][j % ncols].axis("off")

    fig.colorbar(im, ax=axs, shrink=0.6, label="Phase (rad)")
    fig.suptitle("Phase shift (phase_B) vs. tilt angle")
    plt.show()


def plot_LTEM_tilt_series(results, defocus_index=0, cmap="gray"):
    """
    Grid plot of simulated LTEM images across all tilt angles, for one
    chosen defocus value (by index into the defocus_values list used).
    """
    angles = sorted(results.keys())
    defval = results[angles[0]]["defocus_values"][defocus_index]

    all_ims = [results[a]["images"][defocus_index] for a in angles]
    vmin = min(im.min() for im in all_ims)
    vmax = max(im.max() for im in all_ims)

    ncols = min(len(angles), 5)
    nrows = int(np.ceil(len(angles) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    for i, theta in enumerate(angles):
        ax = axs[i // ncols][i % ncols]
        im = ax.imshow(results[theta]["images"][defocus_index], origin="lower",
                        cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"{theta:+.0f}°")
        ax.set_xticks([]); ax.set_yticks([])

    for j in range(len(angles), nrows*ncols):
        axs[j // ncols][j % ncols].axis("off")

    fig.colorbar(im, ax=axs, shrink=0.6, label="Intensity")
    fig.suptitle(f"Simulated LTEM image vs. tilt angle (defocus = {defval:.0f} nm)")
    plt.show()

In [ ]:
# 1. crop + bin (using your previously chosen region/settings)
points_crop, M_crop = crop_points(points, M, x_range=x_range, y_range=y_range)
mags_crop, dx, dy, dz = build_mags_array(points_crop, M_crop)
mags_binned, (dx_b, dy_b, dz_b) = bin_mags(mags_crop, bin_xy=2, bin_z=1,
                                            dx=dx, dy=dy, dz=dz, plot=True)

# 2. sample/beam parameters
sample_params = {
    "B0": 8708,             # Gauss
    "sample_xip0": 100,     # nm, placeholder
    "sample_V0": 0,
    "membrane_thickness": 0,
}
beam_energy = 300e3  # eV

# 3. run the tilt series
angles_deg = np.arange(-60, 61, 10)
results = run_tilt_series(
    mags_binned, dx_b, dz_b, sample_params, beam_energy,
    angles_deg=angles_deg, tilt_axis="x",
    defocus_values=[-1500.87, 1500.87],
)

# 4. plot
plot_phase_tilt_series(results)
plot_LTEM_tilt_series(results, defocus_index=0)  # underfocus
plot_LTEM_tilt_series(results, defocus_index=1)  # overfocus

# one at a time


In [ ]:
sample_params = {
    "B0": 8708,              # Gauss
    "sample_xip0": 100,     # place holder extinction distance
    "sample_V0": 0,         # 
    "membrane_thickness": 0,
}

beam_energy = 300e3  # eV, from Rohan

sim_test = SimLTEM(mags=mags_crop, scale=dx, zscale=dz)
sim_test.set_sample_params(sample_params)

tilt_x=0

import time
t0 = time.time()
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=0, beam_energy=beam_energy)
t1 = time.time()
print(f"Time for cropped volume: {t1 - t0:.2f} sec")
sim_test.show_phase()

In [ ]:
def suggest_defocus(phase, scale, beam_energy, desired_fringe_width_nm=None, verbose=True):
    """
    Estimate a reasonable defocus based on the actual computed phase gradient,
    so Fresnel contrast is visible but doesn't exceed the field of view.

    phase : 2D ndarray, e.g. simulator.phase_B (radians)
    scale : float, in-plane nm/pixel
    beam_energy : float, eV
    desired_fringe_width_nm : float, optional
        Target width of the Fresnel fringe in nm. Defaults to ~10 pixels.
    """
    V = beam_energy  # volts
    lam_nm = 1.22639 / np.sqrt(V + 0.97845e-6 * V**2)  # relativistic electron wavelength, nm

    dphidy, dphidx = np.gradient(phase, scale)  # rad/nm
    grad_mag = np.sqrt(dphidx**2 + dphidy**2)
    max_grad = np.nanmax(grad_mag)

    beta = lam_nm / (2*np.pi) * max_grad  # beam deflection angle, radians

    if desired_fringe_width_nm is None:
        desired_fringe_width_nm = 10 * scale  # default target: ~10 pixels wide

    defocus_nm = desired_fringe_width_nm / beta if beta > 0 else np.inf

    if verbose:
        print(f"Electron wavelength: {lam_nm:.5f} nm")
        print(f"Max phase gradient: {max_grad:.4e} rad/nm")
        print(f"Max beam deflection angle: {beta:.4e} rad")
        print(f"Suggested defocus for ~{desired_fringe_width_nm:.1f} nm fringe: "
              f"{defocus_nm:.0f} nm ({defocus_nm/1000:.2f} um)")

    return defocus_nm

In [ ]:
defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)

In [ ]:
defvals = [defocus_est * f for f in [-0.05,-0.01, -0.005, 0.05]]
defvals = [round(d, -2) for d in defvals]  # round to nearest 100nm for readability
print(defvals)


scope = Microscope(
    E=beam_energy,
    Cs=200e3,
    theta_c=0.01e-3,
    def_spr=80,
)

dd = sim_test.sim_images(defocus_values=defvals, scope=scope, amorphous_bkg=True)



ims = dd.images
fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
vmin, vmax = ims.min(), ims.max()
for i, im in enumerate(ims):
    show_im(im, figax=(fig, axs[i]), cbar=False, ticks_off=True,
            title=f"{defvals[i]:.0f} nm", vmin=vmin, vmax=vmax, scale=sim_test.scale)
plt.tight_layout()
plt.show()

In [ ]:


defvals = [1500.87 ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)


### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)

In [ ]:
sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
).visualize();

In [ ]:
(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

## 20 deg tilt

In [ ]:
tilt_x = 20

tilt_y = 15

sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=tilt_y, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)

In [ ]:

#defvals = [-1438.43  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)


### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)

In [ ]:
sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
).visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

# -20

In [ ]:
tilt_x = -20
tilt_y = 0
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=tilt_y, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)

#defvals = [-1542.74  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)



### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

# 30

In [ ]:
tilt_x = 0
tilt_y =30
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=tilt_y, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)


#defvals = [-1542.74  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)



### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

In [ ]:
tilt_x = -30
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)


#defvals = [-1542.74  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)



### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

# -60

In [ ]:
tilt_x = 60
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)


#defvals = [-1542.74  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)



### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

In [ ]:
tilt_x = -60
sim_test.compute_phase(method="linsup", tilt_x=tilt_x, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)


#defvals = [-1542.74  ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)



### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=f'{tilt_x} deg', vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=f'{tilt_x} deg', scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

# -60

In [ ]:
sim_test.compute_phase(method="linsup", tilt_x=-60, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

defocus_est = suggest_defocus(sim_test.phase_B, scale=sim_test.scale, beam_energy=beam_energy)

In [ ]:
defvals = [-854.37 ] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)


### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=defvals[i], vmin=vmin, vmax=vmax, scale=sim_test.scale)
else:
    show_im(ims, title=defvals[0], scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
)#.visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

In [ ]:
stop

In [ ]:
scope = Microscope(
    E=beam_energy,
    Cs=200e3,
    theta_c=0.01e-3,
    def_spr=80,
)

defvals = [-100_000] # list of one or multiple defocus values
dd = sim_test.sim_images(
    defocus_values=defvals,
    scope=scope,
    amorphous_bkg=True,
)


### plotting
ims = dd.images
if len(defvals) > 1:
    fig, axs = plt.subplots(ncols=len(defvals), figsize=(3*len(defvals), 3))
    vmin, vmax = ims.min(), ims.max()
    for i, im in enumerate(ims):
        show_im(im, figax=(fig,axs[i]), cbar=False, ticks_off=True, title=defvals[i], vmin=vmin, vmax=vmax, scale=sim_linsup.scale)
else:
    show_im(ims, title=defvals[0], scale=sim_test.scale)


sitie = SITIE(
    dd=dd,
    save_dir="./data_dir/452/LTEM_pred",
)

### equivalently, we could do:
# s = SITIE.from_array(
#     image = dd.images[0],
#     scale = dd.scale,
#     defval = dd.defvals[0],
#     beam_energy = dd.beam_energy,
#     simulated = True,
# )

sitie.reconstruct(
    # index=0, # default 0
    # save=True,
).visualize();

(true_By, true_Bx) = sitie.induction_from_phase(sim_test.phase_B)
fig, axs = plt.subplots(ncols=2, figsize=(8,4))
show_2D(true_Bx, true_By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="True B", figax=(fig, axs[0]))
show_2D(sitie.Bx, sitie.By, rad=0, num_arrows=24, arrow_width=0.8, arrow_size=1, title="SITIE B", figax=(fig, axs[1]))
plt.tight_layout()
plt.show()

In [ ]:
sim_test.compute_phase(method="linsup", tilt_x=40, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

In [ ]:
sim_test.compute_phase(method="linsup", tilt_x=60, tilt_y=0, beam_energy=beam_energy)
sim_test.show_phase()

In [ ]:
sim_test.compute_phase(method="linsup", tilt_x=0, tilt_y=20, beam_energy=beam_energy)
sim_test.show_phase()

In [ ]:
sim_test.compute_phase(method="linsup", tilt_x=0, tilt_y=60, beam_energy=beam_energy)
sim_test.show_phase()